# CASMI exploratory data analysis

Run the cells in order with the project Python environment. This notebook uses NumPy, pandas, PyArrow, and Matplotlib. It scans the full labeled training and unlabeled competition files and loads only the columns needed for the plots and follow-up analysis.

Use the existing extracted training file or place the CASMI ZIP in the repo root. If the training file is missing, the loading cell extracts it once to `spectrum_encoder_module/artifacts`, which requires about 3 GB of free space. The competition file is read directly from the ZIP.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq

ROOT = Path.cwd()
ARCHIVE = ROOT / "enveda-CASMI26-molecule-id-mass-spectra.zip"
TRAIN_PATH = ROOT / "spectrum_encoder_module" / "artifacts" / "train.parquet"

metadata_columns = ["precursor_mz", "adduct", "ionization_mode", "instrument_type"]
train_columns = ["inchikey14", "ingest_lib", "molecular_formula", "num_peaks", *metadata_columns]
test_columns = ["molecule_id", "spectrum_id", "ms2_mzs", *metadata_columns]

with ZipFile(ARCHIVE) as archive:
    if not TRAIN_PATH.is_file():
        print(f"Extracting train.parquet to {TRAIN_PATH.parent}")
        archive.extract("train.parquet", TRAIN_PATH.parent)
    test_arrow = pq.read_table(BytesIO(archive.read("test.parquet")), columns=test_columns)

train = pq.read_table(TRAIN_PATH, columns=train_columns).to_pandas(types_mapper=pd.ArrowDtype)
test = (
    test_arrow.append_column("num_peaks", pc.list_value_length(test_arrow["ms2_mzs"]))
    .drop(["ms2_mzs"])
    .to_pandas(types_mapper=pd.ArrowDtype)
)

splits = {"Training": train, "Competition test": test}
molecule_ids = {"Training": "inchikey14", "Competition test": "molecule_id"}
spectra_per_molecule = {
    split: frame[molecule_ids[split]].value_counts()
    for split, frame in splits.items()
}

summary = pd.DataFrame({
    split: {
        "Spectra": len(frame),
        "Molecule groups": len(spectra_per_molecule[split]),
        "Median peaks per spectrum": frame["num_peaks"].median(),
        "Median precursor m/z": frame["precursor_mz"].median(),
        "Median spectra per molecule": spectra_per_molecule[split].median(),
    }
    for split, frame in splits.items()
})
summary.T.round(2)

## Peak counts per spectrum

The curve is the fraction of spectra with **more than** the chosen peak count. The marked 32-peak cap was used in the encoder smoke run. The x-axis is logarithmic.

In [ ]:
peak_counts = {split: frame["num_peaks"].to_numpy() for split, frame in splits.items()}
caps = np.unique(np.r_[np.geomspace(1, 4096, 72).astype(int), 32, 64, 128])
peak_exceedance = pd.DataFrame({
    split: 1 - np.searchsorted(np.sort(counts), caps, side="right") / len(counts)
    for split, counts in peak_counts.items()
}, index=pd.Index(caps, name="Peak count"))

ax = peak_exceedance.plot(figsize=(8, 4), logx=True, linewidth=2)
ax.axvline(32, color="0.4", linestyle="--", linewidth=1)
ax.annotate("32-peak cap", (32, 0.05), xytext=(6, 0), textcoords="offset points")
ax.set(title="Peak counts per spectrum", ylabel="Share above peak count", ylim=(0, 1))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis="y", alpha=0.25)
plt.show()

## Precursor m/z

Each bar uses the total number of spectra in its own split as the denominator. The open-ended final bin includes the high-mass outliers.

In [ ]:
def binned_shares(values_by_split, edges, labels):
    shares = {}
    for split, values in values_by_split.items():
        counts, _ = np.histogram(values, bins=edges)
        shares[split] = counts / len(values)
    return pd.DataFrame(shares, index=labels)

precursor_mz_shares = binned_shares(
    {split: frame["precursor_mz"].to_numpy(dtype=float, na_value=np.nan)
     for split, frame in splits.items()},
    [-np.inf, 200, 250, 300, 350, 400, 450, 500, 600, np.inf],
    ["<200", "200–<250", "250–<300", "300–<350", "350–<400", "400–<450",
     "450–<500", "500–<600", "600+"],
)

ax = precursor_mz_shares.plot.barh(figsize=(8, 5), width=0.8)
ax.invert_yaxis()
ax.set(title="Precursor m/z", xlabel="Share of spectra", ylabel="Precursor m/z bin")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis="x", alpha=0.25)
plt.show()

## Spectra per molecule

Every molecule group counts once. Training groups use `inchikey14`, and competition groups use `molecule_id`.

In [ ]:
molecule_count_shares = binned_shares(
    {split: counts.to_numpy() for split, counts in spectra_per_molecule.items()},
    [1, 2, 3, 4, 5, 6, 10, 20, np.inf],
    ["1", "2", "3", "4", "5", "6–9", "10–19", "20+"],
)

ax = molecule_count_shares.plot.bar(figsize=(8, 4), width=0.8, rot=0)
ax.set(title="Spectra per molecule", xlabel="Spectra per molecule",
       ylabel="Share of molecule groups")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis="y", alpha=0.25)
plt.show()

## Training spectra by source library

These bars show spectrum counts. The competition file has no source library column.

In [ ]:
library_counts = train["ingest_lib"].value_counts()

ax = library_counts.plot.barh(figsize=(8, 5))
ax.invert_yaxis()
ax.set(title="Training spectra by source library", xlabel="Spectra", ylabel="Library")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## Continue exploring

The `train` and `test` DataFrames have one row per spectrum. Edit the example below to explore other metadata, or add fields to `train_columns` and rerun the loading cell. The training frame includes molecule IDs and molecular formulas. The competition frame includes molecule and spectrum IDs.

In [ ]:
library_summary = (
    train.groupby("ingest_lib")
    .agg(
        spectra=("num_peaks", "size"),
        molecules=("inchikey14", "nunique"),
        median_peaks=("num_peaks", "median"),
    )
    .sort_values("spectra", ascending=False)
)
library_summary